# Coach DNA EDA

This notebook explores the cleaned offensive play universe and the first analytical layers used to build team profiles and league baselines.

## Goals
- confirm the cleaned table loads correctly
- inspect season coverage and row counts
- understand the distribution of plays by down, distance, field position, and score state
- review how run/dropback behavior changes across situations
- identify patterns worth carrying into team profiling and scoring

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

candidate_paths = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None

for path in candidate_paths:
    if (path / "python").exists() and (path / "data").exists():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root from notebook.")

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("OUTPUT_TABLES_DIR:", OUTPUT_TABLES_DIR)

In [ ]:
team_baseline_features = pd.read_csv(
    PROCESSED_DATA_DIR / "team_baseline_features_2025_vs_2023_2025.csv"
)

coach_dna_situation_scores = pd.read_csv(
    PROCESSED_DATA_DIR / "coach_dna_situation_scores_2025.csv"
)

coach_dna_team_summary = pd.read_csv(
    PROCESSED_DATA_DIR / "coach_dna_team_summary_2025.csv"
)

print("team_baseline_features:", team_baseline_features.shape)
print("coach_dna_situation_scores:", coach_dna_situation_scores.shape)
print("coach_dna_team_summary:", coach_dna_team_summary.shape)

## 1. Basic Structure Checks
Start by confirming the exported feature and score tables have the expected size and shape.

In [ ]:
team_baseline_features.head()

In [ ]:
coach_dna_situation_scores.head()

In [ ]:
coach_dna_team_summary.head()

In [ ]:
pd.Series({
    "feature_rows": len(team_baseline_features),
    "feature_teams": team_baseline_features["team"].nunique(),
    "feature_situations": team_baseline_features["situation_name"].nunique(),
    "situation_score_rows": len(coach_dna_situation_scores),
    "team_summary_rows": len(coach_dna_team_summary),
})

## 2. Situation Coverage
These checks confirm that every team and every situation are represented the way we expect.

In [ ]:
team_baseline_features.groupby("team").size().sort_values().head(10)

In [ ]:
team_baseline_features.groupby("situation_name").size().sort_values()

In [ ]:
team_baseline_features.groupby("situation_name")["team_play_count"].agg(["count", "min", "median", "mean", "max"]).sort_values("mean")

## 3. Team Sample Size Review
This helps identify which situations are naturally large, small, or noisy.

In [ ]:
sample_summary = (
    team_baseline_features
    .groupby("situation_name", as_index=False)
    .agg(
        avg_team_play_count=("team_play_count", "mean"),
        median_team_play_count=("team_play_count", "median"),
        min_team_play_count=("team_play_count", "min"),
        max_team_play_count=("team_play_count", "max"),
    )
    .sort_values("avg_team_play_count")
)

sample_summary

In [ ]:
team_baseline_features["team_sample_quality"].value_counts()

In [ ]:
team_baseline_features.groupby(["situation_name", "team_sample_quality"]).size().unstack(fill_value=0)

## 4. League Baseline Tendencies by Situation
This section shows how the league tends to behave across the major situations.

In [ ]:
baseline_view = (
    team_baseline_features[
        [
            "situation_order",
            "situation_name",
            "dropback_rate_baseline",
            "rush_rate_baseline",
            "shotgun_rate_baseline",
            "no_huddle_rate_baseline",
            "avg_epa_baseline",
            "success_rate_baseline",
            "explosive_play_rate_baseline",
        ]
    ]
    .drop_duplicates()
    .sort_values("situation_order")
    .reset_index(drop=True)
)

baseline_view

In [ ]:
baseline_view.sort_values("dropback_rate_baseline", ascending=False)[
    ["situation_name", "dropback_rate_baseline", "rush_rate_baseline"]
]

In [ ]:
baseline_view.sort_values("avg_epa_baseline", ascending=False)[
    ["situation_name", "avg_epa_baseline", "success_rate_baseline", "explosive_play_rate_baseline"]
]

## 5. Team Tendency Deviations
Which teams differ most from the league in the way they call games?

In [ ]:
most_run_heavy = (
    team_baseline_features
    .sort_values("rush_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "rush_rate_delta",
            "avg_epa_delta",
            "success_rate_delta",
        ]
    ]
)

most_run_heavy

In [ ]:
most_dropback_heavy = (
    team_baseline_features
    .sort_values("dropback_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "dropback_rate_delta",
            "avg_epa_delta",
            "success_rate_delta",
        ]
    ]
)

most_dropback_heavy

In [ ]:
most_shotgun_heavy = (
    team_baseline_features
    .sort_values("shotgun_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "shotgun_rate_delta",
            "avg_epa_delta",
            "success_rate_delta",
        ]
    ]
)

most_shotgun_heavy

In [ ]:
most_no_huddle_heavy = (
    team_baseline_features
    .sort_values("no_huddle_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "no_huddle_rate_delta",
            "avg_epa_delta",
            "success_rate_delta",
        ]
    ]
)

most_no_huddle_heavy

## 6. Efficiency Deviations
Which teams are outperforming the baseline, and where?

In [ ]:
most_efficient = (
    team_baseline_features
    .sort_values("avg_epa_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "avg_epa_delta",
            "success_rate_delta",
            "explosive_play_rate_delta",
        ]
    ]
)

most_efficient

In [ ]:
least_efficient = (
    team_baseline_features
    .sort_values("avg_epa_delta", ascending=True)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "avg_epa_delta",
            "success_rate_delta",
            "explosive_play_rate_delta",
        ]
    ]
)

least_efficient

In [ ]:
most_successful = (
    team_baseline_features
    .sort_values("success_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "success_rate_delta",
            "avg_epa_delta",
            "explosive_play_rate_delta",
        ]
    ]
)

most_successful

## 7. Explosiveness and Stability
This section checks who creates big plays and who avoids negative outcomes.

In [ ]:
most_explosive = (
    team_baseline_features
    .sort_values("explosive_play_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "explosive_play_rate_delta",
            "avg_epa_delta",
            "success_rate_delta",
        ]
    ]
)

most_explosive

In [ ]:
best_stability = (
    team_baseline_features
    .assign(
        stability_score_proxy=(
            -team_baseline_features["turnover_rate_delta"] - team_baseline_features["sack_rate_delta"]
        )
    )
    .sort_values("stability_score_proxy", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "turnover_rate_delta",
            "sack_rate_delta",
            "avg_epa_delta",
            "success_rate_delta",
        ]
    ]
)

best_stability

In [ ]:
## 8. Team Deep Dive
Use one team code at a time to study how that team differs from league baseline across situations.

In [ ]:
TEAM_CODE = "BUF"

In [ ]:
team_view = (
    team_baseline_features
    .loc[team_baseline_features["team"] == TEAM_CODE]
    .sort_values("situation_order")
)

team_view[
    [
        "team",
        "situation_name",
        "team_play_count",
        "dropback_rate_delta",
        "rush_rate_delta",
        "shotgun_rate_delta",
        "no_huddle_rate_delta",
        "avg_epa_delta",
        "success_rate_delta",
        "explosive_play_rate_delta",
    ]
]

In [ ]:
team_view.sort_values("avg_epa_delta", ascending=False)[
    [
        "situation_name",
        "team_play_count",
        "avg_epa_delta",
        "success_rate_delta",
        "explosive_play_rate_delta",
        "dropback_rate_delta",
        "rush_rate_delta",
    ]
]

In [ ]:
team_view.sort_values("dropback_rate_delta", ascending=False)[
    [
        "situation_name",
        "team_play_count",
        "dropback_rate_delta",
        "rush_rate_delta",
        "avg_epa_delta",
        "success_rate_delta",
    ]
]

## 9. Score Sanity Checks
These checks help verify that the final scores broadly line up with the underlying deltas.

In [ ]:
coach_dna_team_summary.sort_values("overall_coach_dna_score", ascending=False).head(15)

In [ ]:
coach_dna_situation_scores.sort_values("coach_dna_score_adjusted", ascending=False).head(20)[
    [
        "team",
        "situation_name",
        "team_play_count",
        "coach_dna_score_adjusted",
        "tendency_signal_score",
        "efficiency_signal_score",
        "explosiveness_signal_score",
        "stability_signal_score",
        "tendency_profile_label",
        "efficiency_profile_label",
    ]
]

In [ ]:
coach_dna_situation_scores.sort_values("coach_dna_score_adjusted", ascending=True).head(20)[
    [
        "team",
        "situation_name",
        "team_play_count",
        "coach_dna_score_adjusted",
        "tendency_signal_score",
        "efficiency_signal_score",
        "explosiveness_signal_score",
        "stability_signal_score",
        "tendency_profile_label",
        "efficiency_profile_label",
    ]
]

## 10. Draft Findings
Write down early insights worth carrying into the README, GitHub post, or interview narrative.

### Draft findings
- 
- 
- 
- 
- 